# 기본 RAG 파이프라인 구축(항공 기내 규정 매뉴얼 QA 챗봇)

1. 환경 설정

In [16]:
%pip install langchain langchain-community pypdf
%pip install -U langchain-text-splitters
%pip install -U chromadb
%pip install -U langchain-openai faiss-cpu pypdf

In [17]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

2. 문서 로드

In [18]:
# from langchain_community.document_loaders import PyPDFLoader

# loader = PyPDFLoader("company_manual.txt")
# documents = loader.load()

from langchain_community.document_loaders import TextLoader

loader = TextLoader("company_manual.txt", encoding="utf-8")
documents = loader.load()

print(documents)

[Document(metadata={'source': 'company_manual.txt'}, page_content='하늘항공 기내 서비스 및 안전 규정 매뉴얼\nSkyWing Airlines In-Flight Service & Safety Regulation Manual\n문서번호: SW-OPS-2026-04 · 개정판: Rev. 3.1 · 발행일: 2026년 3월\n\n제1장 총칙\n\n1.1 목적\n이 매뉴얼은 하늘항공(가상 항공사, SkyWing Airlines)의 전 노선에서 일관된 기내 서비스 품질과 안전 기준을 유지하기 위해 승무원 및 지상 직원이 준수해야 할 절차를 규정한다.\n\n1.2 적용 범위\n본 규정은 하늘항공이 운항하는 모든 국내선 및 국제선 항공편에 적용되며, 위탁 운항편(코드셰어)에도 별도 계약이 없는 한 동일하게 적용된다.\n\n1.3 용어 정의\n• 위탁 수하물: 탑승 수속 시 항공사에 맡겨 화물칸에 싣는 수하물\n• 기내 수하물: 승객이 객실 내로 직접 휴대하여 반입하는 수하물\n• 제한 품목: 기내 반입은 금지되나 위탁 수하물로는 허용될 수 있는 물품\n• 위험 물품: 위탁 및 기내 반입이 모두 금지되는 물품\n\n제2장 수하물 규정\n\n2.1 위탁 수하물\n일반석 기준 1인당 위탁 수하물은 23kg 이하 1개를 무료로 위탁할 수 있으며, 프레스티지석은 32kg 이하 2개, 퍼스트석은 32kg 이하 3개까지 무료 위탁이 가능하다. 무료 수하물 허용량을 초과하는 경우 초과 수하물 요금이 구간 및 중량에 따라 부과된다.\n\n2.2 기내 수하물\n기내 수하물은 1인당 1개(가로+세로+높이의 합이 115cm 이내, 무게 12kg 이하)와 개인용품 1개(노트북 가방, 핸드백 등)로 제한한다. 상단 선반에 들어가지 않는 크기의 수하물은 탑승구에서 위탁 수하물로 전환 처리한다.\n\n2.3 반려동물 동반 수하물\n반려동물은 이동장을 포함한 총 중량이 7kg 이하인 경우에 한해 기내 동반이 가능하며, 이를 초과하면 화물칸 위탁으로만 

3. 문서 청킹

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'company_manual.txt'}, page_content='하늘항공 기내 서비스 및 안전 규정 매뉴얼\nSkyWing Airlines In-Flight Service & Safety Regulation Manual\n문서번호: SW-OPS-2026-04 · 개정판: Rev. 3.1 · 발행일: 2026년 3월\n\n제1장 총칙\n\n1.1 목적\n이 매뉴얼은 하늘항공(가상 항공사, SkyWing Airlines)의 전 노선에서 일관된 기내 서비스 품질과 안전 기준을 유지하기 위해 승무원 및 지상 직원이 준수해야 할 절차를 규정한다.\n\n1.2 적용 범위\n본 규정은 하늘항공이 운항하는 모든 국내선 및 국제선 항공편에 적용되며, 위탁 운항편(코드셰어)에도 별도 계약이 없는 한 동일하게 적용된다.\n\n1.3 용어 정의\n• 위탁 수하물: 탑승 수속 시 항공사에 맡겨 화물칸에 싣는 수하물\n• 기내 수하물: 승객이 객실 내로 직접 휴대하여 반입하는 수하물\n• 제한 품목: 기내 반입은 금지되나 위탁 수하물로는 허용될 수 있는 물품\n• 위험 물품: 위탁 및 기내 반입이 모두 금지되는 물품\n\n제2장 수하물 규정\n\n2.1 위탁 수하물\n일반석 기준 1인당 위탁 수하물은 23kg 이하 1개를 무료로 위탁할 수 있으며, 프레스티지석은 32kg 이하 2개, 퍼스트석은 32kg 이하 3개까지 무료 위탁이 가능하다. 무료 수하물 허용량을 초과하는 경우 초과 수하물 요금이 구간 및 중량에 따라 부과된다.\n\n2.2 기내 수하물\n기내 수하물은 1인당 1개(가로+세로+높이의 합이 115cm 이내, 무게 12kg 이하)와 개인용품 1개(노트북 가방, 핸드백 등)로 제한한다. 상단 선반에 들어가지 않는 크기의 수하물은 탑승구에서 위탁 수하물로 전환 처리한다.\n\n2.3 반려동물 동반 수하물\n반려동물은 이동장을 포함한 총 중량이 7kg 이하인 경우에 한해 기내 동반이 가능하며, 이를 초과하면 화물칸 위탁으로만 

4. 임베딩 모델

In [20]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

5. 벡터 저장소 구축(Chroma)

In [21]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

6. Retriever - 검색 테스트

In [22]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)

results = retriever.invoke("기내에 가위 반입이 되나요?")

print(results)

[Document(metadata={'source': 'company_manual.txt'}, page_content='2.3 반려동물 동반 수하물\n반려동물은 이동장을 포함한 총 중량이 7kg 이하인 경우에 한해 기내 동반이 가능하며, 이를 초과하면 화물칸 위탁으로만 운송 가능하다. 반려동물 동반은 출발 48시간 전까지 사전 예약이 필요하며, 편당 동반 가능 마리 수는 최대 3마리로 제한된다.\n\n제3장 전자기기 및 배터리 사용 규정\n\n3.1 휴대용 전자기기\n노트북, 태블릿, 스마트폰 등 개인 전자기기는 항시 비행기 탑승모드(에어플레인 모드)로 전환한 상태에서 사용할 수 있다. 이·착륙 시에는 좌석 하단 또는 앞좌석 그물망에 전자기기를 보관해야 한다.\n\n3.2 보조배터리(외장 배터리) 규정\n• 100Wh 이하: 기내 반입 가능, 개인당 최대 5개까지 허용\n• 100Wh 초과 160Wh 이하: 항공사 사전 승인 시에만 반입 가능, 개인당 2개 제한\n• 160Wh 초과: 기내 및 위탁 수하물 모두 반입 금지\n• 모든 보조배터리는 위탁 수하물이 아닌 기내 수하물로만 운송 가능\n\n3.3 전자담배 및 리튬배터리 제품\n전자담배 및 니코틴 흡입 기기는 기내 반입은 가능하나 기내에서의 사용(흡연)은 전 노선에서 금지된다. 위탁 수하물에 넣어 화물칸으로 운송하는 것은 금지된다.\n\n제4장 기내 반입 금지 물품\n다음 물품은 보안 검색 규정에 따라 기내 반입이 금지되며, 적발 시 탑승구 또는 보안 검색대에서 압수될 수 있다.\n• 100ml를 초과하는 액체, 젤, 스프레이류 (단, 투명 지퍼백에 담긴 100ml 이하 용기는 예외)\n• 칼날이 있는 도구 일체 (커터칼, 과도, 가위 등, 단 날 길이 6cm 이하 가위는 위탁 수하물로만 허용)\n• 라이터 및 성냥 (1인당 라이터 1개에 한해 소지 반입 가능, 위탁 수하물 반입은 금지)\n• 스포츠용 배트, 골프채 등 둔기로 사용 가능한 물품\n• 폭죽, 인화성 스프레이 등 위험물로 분류되는 물품\n\n

7. 프롬프트에 검색 결과 결합

In [23]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
당신은 사내 문서를 기반으로 질문에 답변하는 어시스턴트입니다.
아래 제공된 컨텍스트만을 사용하여 질문에 답변하세요.
컨텍스트에 답변할 정보가 없으면 "해당 정보를 찾을 수 없습니다"라고 답하세요.
답변에 사용한 문서의 출처를 명시하세요

컨텍스트:
{context}

질문: {question}

답변:
""")

8. LCEL 체인 구성 (retriever -> context 결합 -> prompt -> llm -> 문자열 출력)

In [24]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = chain.invoke("기내에 반입이 안되는 물품은 어떤것이 있나요?")

print(answer)


기내에 반입이 안되는 물품은 다음과 같습니다:
- 100ml를 초과하는 액체, 젤, 스프레이류 (단, 투명 지퍼백에 담긴 100ml 이하 용기는 예외)
- 칼날이 있는 도구 일체 (커터칼, 과도, 가위 등, 단 날 길이 6cm 이하 가위는 위탁 수하물로만 허용)
- 라이터 및 성냥 (1인당 라이터 1개에 한해 소지 반입 가능, 위탁 수하물 반입은 금지)
- 스포츠용 배트, 골프채 등 둔기로 사용 가능한 물품
- 폭죽, 인화성 스프레이 등 위험물로 분류되는 물품

출처: 제4장 기내 반입 금지 물품


9. 간단한 QA 챗봇 루프(답변 + 참고한 문서 조각을 함께 출력)

In [25]:
def ask_with_source(question: str):
  docs = retriever.invoke(question)
  answer = chain.invoke(question)

  print(f"답변: {answer}\n")
  print(f"--- 참고한 문서 조각 ({len(docs)}개) ---")
  for i, doc in enumerate(docs, 1):
      page = doc.metadata.get("page", "?")
      print(f"[{i}] (p.{page}) {doc.page_content[:200]}...")
  print()

print("항공 RAG 챗봇 준비 완료! 종료하려면 'quit' 입력.\n")

while True:
  question = input("질문 : ")
  if question.strip().lower() in ("quit", "exit"):
    print("챗봇을 종료합니다.")
    break
  ask_with_source(question)

항공 RAG 챗봇 준비 완료! 종료하려면 'quit' 입력.

질문 : 기내에 강아지 데리고 타도 돼?
답변: 기내에 강아지를 데리고 타려면 이동장을 포함한 총 중량이 7kg 이하이어야 하며, 출발 48시간 전까지 사전 예약이 필요합니다. 또한, 편당 동반 가능 마리 수는 최대 3마리로 제한됩니다. (출처: 2.3 반려동물 동반 수하물)

--- 참고한 문서 조각 (1개) ---
[1] (p.?) 2.3 반려동물 동반 수하물
반려동물은 이동장을 포함한 총 중량이 7kg 이하인 경우에 한해 기내 동반이 가능하며, 이를 초과하면 화물칸 위탁으로만 운송 가능하다. 반려동물 동반은 출발 48시간 전까지 사전 예약이 필요하며, 편당 동반 가능 마리 수는 최대 3마리로 제한된다.

제3장 전자기기 및 배터리 사용 규정

3.1 휴대용 전자기기
노트북, 태블릿,...

질문 : 기내에 음료 들고 타도 돼?
답변: 해당 정보를 찾을 수 없습니다.

--- 참고한 문서 조각 (1개) ---
[1] (p.?) 2.3 반려동물 동반 수하물
반려동물은 이동장을 포함한 총 중량이 7kg 이하인 경우에 한해 기내 동반이 가능하며, 이를 초과하면 화물칸 위탁으로만 운송 가능하다. 반려동물 동반은 출발 48시간 전까지 사전 예약이 필요하며, 편당 동반 가능 마리 수는 최대 3마리로 제한된다.

제3장 전자기기 및 배터리 사용 규정

3.1 휴대용 전자기기
노트북, 태블릿,...

질문 : 4kg인 고양이 기내에 데리고 타도 돼?
답변: 네, 4kg인 고양이는 이동장을 포함한 총 중량이 7kg 이하이므로 기내에 동반할 수 있습니다. 단, 출발 48시간 전까지 사전 예약이 필요하며, 편당 동반 가능 마리 수는 최대 3마리로 제한됩니다. 

출처: 2.3 반려동물 동반 수하물

--- 참고한 문서 조각 (1개) ---
[1] (p.?) 2.3 반려동물 동반 수하물
반려동물은 이동장을 포함한 총 중량이 7kg 이하인 경우에 한해 기내 동반이 가능하며, 이를 초과하면 화물칸 위탁으로만 운송 가능하